# NPU 병렬 추론 — 멀티코어(async) & 멀티카드(round-robin)

같은 배치(N장)를 3가지로 처리하며 처리량 비교.
**코어(한 장 8개)는 async가 자동 병렬, 카드(여러 장)는 라운드로빈으로 직접 분산.**
> 커널: `pe_npu_host`. 상세 실측: `../../reports/performance/NPU_multicard_62ch_benchmark.md`

In [ ]:
import os, sys, glob, time
import numpy as np
sys.path.insert(0, os.path.abspath("../.."))
import pe_npu
from pe_npu import assets
import qbruntime

def detect_npus():
    return sorted(int(os.path.basename(p)[5:]) for p in glob.glob("/dev/aries*") if os.path.basename(p)[5:].isdigit())
NPUS = detect_npus(); BATCH = 32
mxq = assets.ensure_full_mxq(scheme="single")
img = sorted(glob.glob("images/*.jpg"))
chw = pe_npu.preprocess_image(img[0]) if img else np.random.randn(3,336,336).astype(np.float32)
X = np.ascontiguousarray(np.asarray(chw).transpose(1,2,0).astype(np.float32))
print(f"장착 NPU: {NPUS} ({len(NPUS)}장) | 배치 {BATCH}장")

## 1. sync (1카드 직렬) vs async (1카드 8코어) vs multicard (전 카드)

In [ ]:
import statistics
def med(fn,n=5):
    ts=[]
    for _ in range(n): t=time.perf_counter(); fn(); ts.append((time.perf_counter()-t)*1000)
    return statistics.median(ts)

# 1) sync
ms=qbruntime.Model(mxq); ms.launch(qbruntime.Accelerator(NPUS[0])); ms.infer(X)
t_sync=med(lambda:[ms.infer(X) for _ in range(BATCH)]); ms.dispose()
# 2) async (8코어)
cfg=qbruntime.ModelConfig(); cfg.set_async_pipeline_enabled(True)
ma=qbruntime.Model(mxq,cfg); ma.launch(qbruntime.Accelerator(NPUS[0])); ma.infer_async(X).get()
def run_a():
    fs=[ma.infer_async(X) for _ in range(BATCH)]; [f.get() for f in fs]
t_async=med(run_a); ma.dispose()
# 3) multicard
mm=[]
for d in NPUS:
    c=qbruntime.ModelConfig(); c.set_async_pipeline_enabled(True)
    m=qbruntime.Model(mxq,c); m.launch(qbruntime.Accelerator(d)); mm.append(m)
for m in mm: m.infer_async(X).get()
def run_m():
    fs=[mm[i%len(mm)].infer_async(X) for i in range(BATCH)]; [f.get() for f in fs]
t_multi=med(run_m); [m.dispose() for m in mm]

for name,t in [("sync",t_sync),("async(8core)",t_async),(f"multicard({len(NPUS)})",t_multi)]:
    print(f"{name:16s}: {t:7.1f} ms  ({BATCH/(t/1000):6.1f} img/s)  x{t_sync/t:.1f}")

## 2. 처리량 시각화

In [ ]:
import matplotlib.pyplot as plt
labels=["sync","async\n(8 core)",f"multicard\n({len(NPUS)} cards)"]
ips=[BATCH/(t_sync/1000), BATCH/(t_async/1000), BATCH/(t_multi/1000)]
fig,ax=plt.subplots(figsize=(6,4))
b=ax.bar(labels, ips, color=["#9aa0a6","#457b9d","#2a9d8f"])
for r,v in zip(b,ips): ax.text(r.get_x()+r.get_width()/2, v+max(ips)*0.01, f"{v:.1f}", ha="center")
ax.set_ylabel("Throughput (img/s)"); ax.set_title(f"NPU parallel inference (batch={BATCH})")
plt.tight_layout(); plt.show()
print("코어=async 자동 병렬, 카드=라운드로빈 직접 분산")